# Observabilidad, Metadata y Linaje para Ecosistemas IA/Big Data (Simulación)

Objetivo: simular una tubería de datos con eventos de cambio (schema, frescura, errores), modelar linaje de datasets y jobs, y generar un **reporte de impacto** para gobernanza y auditoría.

Cubrimos: métricas (freshness, latency, error-rate), logs, alertas; gestión de metadatos mínimos; linaje table-level/column-level (simplificado).

## 1. Modelo simple de linaje y metadatos
Representamos datasets, jobs y dependencias. Incluimos columnas relevantes por dataset para un trazo de impacto por cambios de esquema.

In [ ]:
from dataclasses import dataclass, field
from typing import Dict, List, Set
import pandas as pd

@dataclass
class Dataset:
    name: str
    columns: Set[str] = field(default_factory=set)
    domain: str = ''
    owner: str = ''

@dataclass
class Job:
    name: str
    inputs: List[str]
    outputs: List[str]
    code_ref: str = ''  # notebook/file

class Lineage:
    def __init__(self):
        self.datasets: Dict[str, Dataset] = {}
        self.jobs: Dict[str, Job] = {}
        self.parents: Dict[str, Set[str]] = {}   # dataset -> parents datasets
        self.children: Dict[str, Set[str]] = {}  # dataset -> children datasets
    def add_dataset(self, ds: Dataset):
        self.datasets[ds.name] = ds
        self.parents.setdefault(ds.name, set())
        self.children.setdefault(ds.name, set())
    def add_job(self, job: Job):
        self.jobs[job.name] = job
        for o in job.outputs:
            self.parents.setdefault(o, set()).update(job.inputs)
        for i in job.inputs:
            self.children.setdefault(i, set()).update(job.outputs)
    def impact_upstream(self, dataset: str) -> Set[str]:
        # retorna todos los downstream afectados por cambios en dataset
        impacted = set()
        stack = [dataset]
        while stack:
            cur = stack.pop()
            for ch in self.children.get(cur, []):
                if ch not in impacted:
                    impacted.add(ch)
                    stack.append(ch)
        return impacted
    def find_dependents_by_column(self, source_ds: str, column: str) -> List[str]:
        # detecta datasets descendientes que declaran depender de la columna
        impacted = []
        for ds_name in self.impact_upstream(source_ds):
            ds = self.datasets.get(ds_name)
            if ds and column in ds.columns:
                impacted.append(ds_name)
        return impacted

# Definimos un pipeline típico IA/Big Data
L = Lineage()

# Bronze
L.add_dataset(Dataset('bronze.sales_orders_raw', {'order_id','order_amount','order_date','customer_id'},'sales','owner_sales'))
L.add_dataset(Dataset('bronze.marketing_campaigns_raw', {'campaign_id','channel','impressions','clicks','spend'},'marketing','owner_marketing'))

# Jobs a Silver
L.add_job(Job('clean_sales_orders',[ 'bronze.sales_orders_raw' ],[ 'silver.orders_clean' ], code_ref='02-nivel-mid/18_observabilidad_metadata_linaje_ia_bigdata_simulacion.ipynb'))
L.add_job(Job('clean_marketing_campaigns',[ 'bronze.marketing_campaigns_raw' ],[ 'silver.campaigns_clean' ], code_ref='02-nivel-mid/18_observabilidad_metadata_linaje_ia_bigdata_simulacion.ipynb'))

# Silver datasets
L.add_dataset(Dataset('silver.orders_clean', {'order_id','order_amount','order_date','customer_id','channel'}))
L.add_dataset(Dataset('silver.campaigns_clean', {'campaign_id','channel','impressions','clicks','spend'}))

# Gold y features
L.add_job(Job('join_sales_campaigns',[ 'silver.orders_clean','silver.campaigns_clean' ],[ 'gold.bi_sales_campaign_join' ], code_ref='02-nivel-mid/18_observabilidad_metadata_linaje_ia_bigdata_simulacion.ipynb'))
L.add_dataset(Dataset('gold.bi_sales_campaign_join', {'order_id','order_amount','order_date','customer_id','channel','campaign_id','impressions','clicks','spend'}))

L.add_job(Job('build_orders_features',[ 'silver.orders_clean' ],[ 'features.orders_features' ], code_ref='02-nivel-mid/18_observabilidad_metadata_linaje_ia_bigdata_simulacion.ipynb'))
L.add_dataset(Dataset('features.orders_features', {'customer_id','orders_30d','revenue_30d'}))

# Model consumer
L.add_job(Job('train_model_reco',[ 'features.orders_features' ],[ 'models.reco_v1' ], code_ref='02-nivel-mid/18_observabilidad_metadata_linaje_ia_bigdata_simulacion.ipynb'))
L.add_dataset(Dataset('models.reco_v1', {'model_artifact','metrics_auc'}))

print('Datasets:', len(L.datasets), '| Jobs:', len(L.jobs))

## 2. Eventos operacionales y métricas (observabilidad)
Simulamos freshness (minutos), error-rate, throughput y generamos alertas por umbral.

In [ ]:
from datetime import datetime, timedelta
import random
random.seed(7)

metrics = {
    'bronze.sales_orders_raw': {'freshness_m': 20, 'error_rate': 0.002, 'throughput_rps': 120},
    'bronze.marketing_campaigns_raw': {'freshness_m': 75, 'error_rate': 0.0, 'throughput_rps': 90},
    'silver.orders_clean': {'freshness_m': 25, 'error_rate': 0.005, 'throughput_rps': 300},
    'silver.campaigns_clean': {'freshness_m': 80, 'error_rate': 0.0, 'throughput_rps': 220},
    'gold.bi_sales_campaign_join': {'freshness_m': 85, 'error_rate': 0.01, 'throughput_rps': 150},
}

thresholds = { 'freshness_m_max': 60, 'error_rate_max': 0.01 }

alerts = []
for ds, m in metrics.items():
    if m['freshness_m'] > thresholds['freshness_m_max']:
        alerts.append((ds, 'freshness', m['freshness_m']))
    if m['error_rate'] > thresholds['error_rate_max']:
        alerts.append((ds, 'error_rate', m['error_rate']))

print('Alertas generadas:', alerts)

## 3. Evento de cambio de esquema y reporte de impacto
Caso: renombramos `order_amount` → `amount` en `bronze.sales_orders_raw`. Detectamos descendientes afectados (column-level simplificado).

In [ ]:
# Cambio de esquema en la fuente
source = 'bronze.sales_orders_raw'
old_col, new_col = 'order_amount', 'amount'
print('Schema change:', source, old_col, '→', new_col)

# Aplicamos el cambio en el dataset fuente
if old_col in L.datasets[source].columns:
    L.datasets[source].columns.remove(old_col)
    L.datasets[source].columns.add(new_col)

# Descendientes que dependen explícitamente de la columna antigua
dependientes = L.find_dependents_by_column(source, old_col)
report = pd.DataFrame({'impacted_dataset': dependientes})
report

### 3.1 Recomendaciones automáticas
- Actualizar transformaciones en `clean_sales_orders` y aguas abajo.
- Agregar regla de compatibilidad (alias `order_amount`→`amount`) temporal.
- Notificar owners de `silver.orders_clean`, `gold.bi_sales_campaign_join`, `features.orders_features`.

## 4. Linaje y análisis de impacto por frescura
Cuando `bronze.marketing_campaigns_raw` supera frescura de 60m, los consumidores `silver.campaigns_clean` y `gold.bi_sales_campaign_join` heredan el retraso.

In [ ]:
stale_source = 'bronze.marketing_campaigns_raw'
down = L.impact_upstream(stale_source)
impact_fresh = pd.DataFrame({
    'downstream': list(sorted(down)),
    'note': 'posible retraso por fuente con freshness>60m'
})
impact_fresh

## 5. Logs y auditoría mínima
Registramos eventos con timestamp, tipo, severidad y entidad afectada (dataset/job).

In [ ]:
from datetime import datetime
logs = []
def log(event_type, severity, entity, details):
    logs.append({'ts': datetime.utcnow().isoformat(), 'type': event_type, 'sev': severity, 'entity': entity, 'details': details})

log('schema_change','high', source, f'{old_col}-> {new_col}')
for ds, kind, val in alerts:
    log('alert', 'medium', ds, f'{kind}={val}')
pd.DataFrame(logs)

## 6. Reporte final (gobernanza)
Incluye: alertas activas, datasets impactados por esquema y por frescura, y próximos pasos.

In [ ]:
reporte = {
  'alerts': alerts,
  'schema_change_source': source,
  'schema_impacted': dependientes,
  'freshness_source': stale_source,
  'freshness_impacted': list(sorted(down)),
  'recommendations': [
      'Agregar alias de columna y deprecación programada',
      'Actualizar contratos de datos y avisar a consumidores',
      'Asegurar pruebas de compatibilidad y dashboards de frescura'
  ]
}
pd.Series(reporte)